## 1. Load data
We load the EV charging dataset and inspect the first rows.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("EVChargingStationUsage.csv", low_memory=False)

print("Shape:", df.shape)
display(df.head())


Shape: (259415, 33)


,Station Name,MAC Address,Org Name,Start Date,Start Time Zone,End Date,End Time Zone,Transaction Date (Pacific Time),Total Duration (hh:mm:ss),Charging Time (hh:mm:ss),...,Longitude,Currency,Fee,Ended By,Plug In Event Id,Driver Postal Code,User ID,County,System S/N,Model Number
0,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/29/2011 20:17,PDT,7/29/2011 23:20,PDT,7/29/2011 23:20,3:03:32,1:54:03,...,-122.160309,USD,0.0,Plug Out at Vehicle,3,95124.0,3284,NaN,NaN,NaN
1,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/30/2011 0:00,PDT,7/30/2011 0:02,PDT,7/30/2011 0:02,0:02:06,0:01:54,...,-122.160309,USD,0.0,Customer,4,94301.0,4169,NaN,NaN,NaN
2,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/30/2011 8:16,PDT,7/30/2011 12:34,PDT,7/30/2011 12:34,4:17:32,4:17:28,...,-122.160309,USD,0.0,Plug Out at Vehicle,5,94301.0,4169,NaN,NaN,NaN
3,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/30/2011 14:51,PDT,7/30/2011 16:55,PDT,7/30/2011 16:55,2:03:24,2:02:58,...,-122.160309,USD,0.0,Customer,6,94302.0,2545,NaN,NaN,NaN
4,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/30/2011 18:51,PDT,7/30/2011 20:03,PDT,7/30/2011 20:03,1:11:24,0:43:54,...,-122.160309,USD,0.0,Plug Out at Vehicle,7,94043.0,3765,NaN,NaN,NaN


## 2. Basic quality checks
We check missing values, duplicates, and data types.

In [2]:
print("Missing values per column:")
display(df.isna().sum().sort_values(ascending=False))

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nData types:")
display(df.dtypes)

Missing values per column:


County                             84665
EVSE ID                            78948
System S/N                         78948
Model Number                       78948
Driver Postal Code                  8402
User ID                             7677
Currency                            1788
Ended By                             248
Transaction Date (Pacific Time)      209
Port Type                              9
Start Time Zone                        0
Postal Code                            0
Org Name                               0
Start Date                             0
Plug In Event Id                       0
Fee                                    0
Longitude                              0
Latitude                               0
Country                                0
City                                   0
State/Province                         0
End Date                               0
Address 1                              0
MAC Address                            0
Plug Type       


Exact duplicate rows: 4

Data types:


Station Name                           str
MAC Address                            str
Org Name                               str
Start Date                             str
Start Time Zone                        str
End Date                               str
End Time Zone                          str
Transaction Date (Pacific Time)        str
Total Duration (hh:mm:ss)              str
Charging Time (hh:mm:ss)               str
Energy (kWh)                       float64
GHG Savings (kg)                   float64
Gasoline Savings (gallons)         float64
Port Type                              str
Port Number                          int64
Plug Type                              str
EVSE ID                            float64
Address 1                              str
City                                   str
State/Province                         str
Postal Code                          int64
Country                                str
Latitude                           float64
Longitude  

## 3. Clean text columns
We remove extra spaces and normalize station names.

In [3]:
text_cols = ["Station Name", "Port Type", "Plug Type", "Ended By", "Start Time Zone", "End Time Zone"]

for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

df["Station Name"] = (
    df["Station Name"]
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"\s*#\s*", " #", regex=True)
)

df["Start Time Zone"] = df["Start Time Zone"].str.upper()
df["End Time Zone"] = df["End Time Zone"].str.upper()

## 4. Parse dates and durations
We convert date/time columns and duration strings into proper formats.

In [4]:
df["Start Date"] = pd.to_datetime(df["Start Date"], errors="coerce")
df["End Date"] = pd.to_datetime(df["End Date"], errors="coerce")
df["Transaction Date (Pacific Time)"] = pd.to_datetime(df["Transaction Date (Pacific Time)"], errors="coerce")

total_duration = pd.to_timedelta(df["Total Duration (hh:mm:ss)"], errors="coerce")
charging_duration = pd.to_timedelta(df["Charging Time (hh:mm:ss)"], errors="coerce")

## 5. Convert numeric fields
We convert important numeric columns to numeric values.

In [5]:
numeric_cols = ["Energy (kWh)", "Fee", "Latitude", "Longitude", "GHG Savings (kg)", "Gasoline Savings (gallons)"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

## 6. Build cleaned session dataset
We remove exact duplicates and keep sessions with valid start/end timestamps.

In [6]:
invalid_time = (
    df["Start Date"].isna() |
    df["End Date"].isna() |
    (df["End Date"] <= df["Start Date"])
)

timestamp_duration_min = (df["End Date"] - df["Start Date"]).dt.total_seconds() / 60
total_duration_min = total_duration.dt.total_seconds() / 60
duration_diff_min = (timestamp_duration_min - total_duration_min).abs()

print("Invalid time rows:", invalid_time.sum())
print("Rows with duration mismatch > 2 min:", (duration_diff_min > 2).sum())
print("Negative energy rows:", (df["Energy (kWh)"] < 0).sum())

df = df.drop_duplicates().copy()

clean_mask = (
    df["Start Date"].notna() &
    df["End Date"].notna() &
    (df["End Date"] > df["Start Date"]) &
    df["Station Name"].notna()
)

df_clean = df.loc[clean_mask].copy()

print("\nRows after duplicate removal:", len(df))
print("Rows in cleaned dataset:", len(df_clean))

Invalid time rows: 83
Rows with duration mismatch > 2 min: 454
Negative energy rows: 0

Rows after duplicate removal: 259411
Rows in cleaned dataset: 259328


## 7. Convert timestamps to UTC
We convert start/end times to UTC usind their timezone labels.

In [8]:
zone_map = {"PST": "Etc/GMT+8", "PDT": "Etc/GMT+7", "UTC": "UTC"}

df_clean["Start_UTC"] = pd.Series(pd.NaT, index=df_clean.index, dtype="datetime64[ns, UTC]")
df_clean["End_UTC"] = pd.Series(pd.NaT, index=df_clean.index, dtype="datetime64[ns, UTC]")

for label, tz in zone_map.items():
    m1 = (df_clean["Start Time Zone"] == label) & df_clean["Start Date"].notna()
    df_clean.loc[m1, "Start_UTC"] = (
        df_clean.loc[m1, "Start Date"]
        .dt.tz_localize(tz)
        .dt.tz_convert("UTC")
    )

    m2 = (df_clean["End Time Zone"] == label) & df_clean["End Date"].notna()
    df_clean.loc[m2, "End_UTC"] = (
        df_clean.loc[m2, "End Date"]
        .dt.tz_localize(tz)
        .dt.tz_convert("UTC")
    )

df_clean["Elapsed_UTC_Minutes"] = (df_clean["End_UTC"] - df_clean["Start_UTC"]).dt.total_seconds() / 60

recorded_minutes = total_duration.loc[df_clean.index].dt.total_seconds() / 60
df_clean["UTC_Duration_Difference_Minutes"] = (df_clean["Elapsed_UTC_Minutes"] - recorded_minutes).abs()

display(
    df_clean[
        ["Start Date", "End Date", "Start_UTC", "End_UTC", "Elapsed_UTC_Minutes", "UTC_Duration_Difference_Minutes"]
    ].head()
)

,Start Date,End Date,Start_UTC,End_UTC,Elapsed_UTC_Minutes,UTC_Duration_Difference_Minutes
0,2011-07-29 20:17:00,2011-07-29 23:20:00,2011-07-30 03:17:00+00:00,2011-07-30 06:20:00+00:00,183.0,0.533333
1,2011-07-30 00:00:00,2011-07-30 00:02:00,2011-07-30 07:00:00+00:00,2011-07-30 07:02:00+00:00,2.0,0.100000
2,2011-07-30 08:16:00,2011-07-30 12:34:00,2011-07-30 15:16:00+00:00,2011-07-30 19:34:00+00:00,258.0,0.466667
3,2011-07-30 14:51:00,2011-07-30 16:55:00,2011-07-30 21:51:00+00:00,2011-07-30 23:55:00+00:00,124.0,0.600000
4,2011-07-30 18:51:00,2011-07-30 20:03:00,2011-07-31 01:51:00+00:00,2011-07-31 03:03:00+00:00,72.0,0.600000


## 8. Final time-quality filter
We keep only rows that are valid for time-based analysis.

In [9]:
time_quality_mask = (
    df_clean["Start_UTC"].notna() &
    df_clean["End_UTC"].notna() &
    (df_clean["Elapsed_UTC_Minutes"] > 0) &
    (
        df_clean["UTC_Duration_Difference_Minutes"].isna() |
        (df_clean["UTC_Duration_Difference_Minutes"] <= 2)
    )
)

df_time_clean = df_clean.loc[time_quality_mask].copy()
df_time_review = df_clean.loc[~time_quality_mask].copy()

print("Rows kept for analysis:", len(df_time_clean))
print("Rows sent to review:", len(df_time_review))

Rows kept for analysis: 259002
Rows sent to review: 326


## 9. Helper function: split one session into hours
We split each charging session into hourly overlaps.

In [10]:
def split_session_into_hours(start_utc, end_utc):
    rows = []
    h = start_utc.floor("h")

    while h < end_utc:
        h_end = h + pd.Timedelta(hours=1)
        overlap_start = max(start_utc, h)
        overlap_end = min(end_utc, h_end)

        overlap_hours = (overlap_end - overlap_start).total_seconds() / 3600
        if overlap_hours > 0:
            rows.append((h, overlap_hours))

        h = h_end

    return rows

## 10. Build hourly station and user tables
Energy is allocated to each hour proportionally to overlap duration.

In [11]:
station_rows = []
user_rows = []

for _, r in df_time_clean.iterrows():
    parts = split_session_into_hours(r["Start_UTC"], r["End_UTC"])
    total_h = sum(x[1] for x in parts)

    energy = r["Energy (kWh)"] if pd.notna(r["Energy (kWh)"]) else 0.0

    for hour, occ_h in parts:
        e_h = (energy * occ_h / total_h) if total_h > 0 else 0.0

        station_rows.append({
            "Station Name": r["Station Name"],
            "Hour": hour,
            "Energy (kWh)": e_h,
            "Occupied_Port_Hours": occ_h,
            "Session_Count": 1
        })

        if pd.notna(r["User ID"]) and str(r["User ID"]).strip() != "":
            user_rows.append({
                "User ID": str(r["User ID"]).strip(),
                "Hour": hour,
                "Energy (kWh)": e_h,
                "Occupied_Port_Hours": occ_h,
                "Session_Count": 1
            })

hourly_energy = (
    pd.DataFrame(station_rows)
    .groupby(["Station Name", "Hour"], as_index=False)
    .sum()
)

hourly_user_usage = (
    pd.DataFrame(user_rows)
    .groupby(["User ID", "Hour"], as_index=False)
    .sum()
)

print("hourly_energy rows:", len(hourly_energy))
print("hourly_user_usage rows:", len(hourly_user_usage))
display(hourly_energy.head())

hourly_energy rows: 653425
hourly_user_usage rows: 861442


,Station Name,Hour,Energy (kWh),Occupied_Port_Hours,Session_Count
0,PALO ALTO CA / BRYANT #1,2011-10-13 19:00:00+00:00,0.664281,0.183333,1
1,PALO ALTO CA / BRYANT #1,2011-10-13 20:00:00+00:00,0.966227,0.266667,1
2,PALO ALTO CA / BRYANT #1,2011-10-14 01:00:00+00:00,0.681748,0.183333,1
3,PALO ALTO CA / BRYANT #1,2011-10-14 02:00:00+00:00,3.718623,1.000000,1
4,PALO ALTO CA / BRYANT #1,2011-10-14 03:00:00+00:00,2.231174,0.600000,1


## 11. Add local-time features
We create local hour/day fields for behavior analysis.

In [12]:
hourly_energy["Hour_Local"] = hourly_energy["Hour"].dt.tz_convert("America/Los_Angeles")
hourly_energy["Hour_Of_Day"] = hourly_energy["Hour_Local"].dt.hour
hourly_energy["Weekday_Local"] = hourly_energy["Hour_Local"].dt.dayofweek

hourly_user_usage["Hour_Local"] = hourly_user_usage["Hour"].dt.tz_convert("America/Los_Angeles")
hourly_user_usage["Hour_Of_Day"] = hourly_user_usage["Hour_Local"].dt.hour
hourly_user_usage["Weekday_Local"] = hourly_user_usage["Hour_Local"].dt.dayofweek

display(hourly_energy.head())
display(hourly_user_usage.head())

/var/folders/v_/70wdhjs53ylbyj_44k3d8p4r0000gn/T/ipykernel_74168/3331452947.py:3: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  hourly_energy["Weekday_Local"] = hourly_energy["Hour_Local"].dt.dayofweek
/var/folders/v_/70wdhjs53ylbyj_44k3d8p4r0000gn/T/ipykernel_74168/3331452947.py:7: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  hourly_user_usage["Weekday_Local"] = hourly_user_usage["Hour_Local"].dt.dayofweek


,Station Name,Hour,Energy (kWh),Occupied_Port_Hours,Session_Count,Hour_Local,Hour_Of_Day,Weekday_Local
0,PALO ALTO CA / BRYANT #1,2011-10-13 19:00:00+00:00,0.664281,0.183333,1,2011-10-13 12:00:00-07:00,12,3
1,PALO ALTO CA / BRYANT #1,2011-10-13 20:00:00+00:00,0.966227,0.266667,1,2011-10-13 13:00:00-07:00,13,3
2,PALO ALTO CA / BRYANT #1,2011-10-14 01:00:00+00:00,0.681748,0.183333,1,2011-10-13 18:00:00-07:00,18,3
3,PALO ALTO CA / BRYANT #1,2011-10-14 02:00:00+00:00,3.718623,1.000000,1,2011-10-13 19:00:00-07:00,19,3
4,PALO ALTO CA / BRYANT #1,2011-10-14 03:00:00+00:00,2.231174,0.600000,1,2011-10-13 20:00:00-07:00,20,3


,User ID,Hour,Energy (kWh),Occupied_Port_Hours,Session_Count,Hour_Local,Hour_Of_Day,Weekday_Local
0,0,2011-08-02 19:00:00+00:00,1.959811,0.566667,1,2011-08-02 12:00:00-07:00,12,1
1,0,2011-08-02 20:00:00+00:00,1.556320,0.450000,1,2011-08-02 13:00:00-07:00,13,1
2,0,2011-08-05 03:00:00+00:00,2.307848,0.783333,1,2011-08-04 20:00:00-07:00,20,3
3,0,2011-08-05 04:00:00+00:00,2.946189,1.000000,1,2011-08-04 21:00:00-07:00,21,3
4,0,2011-08-05 05:00:00+00:00,2.946189,1.000000,1,2011-08-04 22:00:00-07:00,22,3
